# C10 prereg42：訓練配方短跑 V2（Colab）

**不用改任何程式**：執行階段選 A100（或其他 GPU），按「執行階段 → 全部執行」。

1. 掛上雲端硬碟，核對 `我的雲端硬碟/C10/p42/C10_p42_kit.zip` 的 SHA-256，解開並逐檔核對。
2. 訓練 V2 40,000 步（約 3 小時；每 8,000 步存到 `C10/p42/V2/`；斷線後重按「全部執行」會接著跑）。
3. 算 D 玩家的向量（raw、EMA）存到 `C10/p42/emb/`。
4. 打包 `C10/p42/C10_p42_V2_results.zip`，下載回傳。

任何一格出現紅字或 `STOP`，停下來，截最後 20 行。

In [ ]:
# 0. 把 Drive 上 C10/p42/kit_parts/ 的 66 份接回 C10_p42_kit.zip（已存在且雜湊對就略過）
import hashlib, os, glob
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/C10/p42'
KIT = os.path.join(D, 'C10_p42_kit.zip')
SHA = '6f5c93650375db2d13a3182d5c2808902b679686975438486a0342e5ad5a7306'
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()
if not (os.path.exists(KIT) and sha(KIT) == SHA):
    parts = sorted(glob.glob(os.path.join(D, 'kit_parts', 'kit.part*')))
    assert len(parts) == 66, 'STOP: kit_parts 只有 %d 份' % len(parts)
    tmp = '/content/C10_p42_kit.zip'
    with open(tmp, 'wb') as o:
        for p in parts:
            with open(p, 'rb') as f:
                o.write(f.read())
    assert sha(tmp) == SHA, 'STOP: 接回的 zip 雜湊不對'
    import shutil; shutil.copyfile(tmp, KIT)
    assert sha(KIT) == SHA, 'STOP: 複製到 Drive 後雜湊不對'
print('kit OK', SHA[:12])


In [ ]:
# 1. 雲端硬碟、GPU、套件（不用改）
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE, WORK, VARIANT = '/content/drive/MyDrive/C10/p42', '/content/c10', 'V2'
KIT_SHA, FILES = '6f5c93650375db2d13a3182d5c2808902b679686975438486a0342e5ad5a7306', {"data/prereg06/feats/meta.npz": "b1b5def7312dc6321ab08dd7e74e9b9069c49e9efb8caf7133b9869bf7338df5", "data/prereg36/cache/cache.json": "764a61937e60ab660972ecbe8a585e4205bd27a132ce74a64f4cc7390533c48a", "data/prereg36/cache/codes.npy": "be5831abbccf22be3c96725082939638e6607f88ad45b1ed2164db33060c0303", "data/prereg36/cache/meta.npz": "fa8bf685be64aa100746238caeaa87f917d6adc194d2026d83c7865f61f834d3", "data/prereg36/cache/off.npy": "7c352279359c5035c1693b78a783d534c8b6ac1e7f870e56655b4b764fb2eaf8", "results/prereg36/explore/p36_embed.py": "2f2e732d1309e25e0b70e370583c07b74dd4c692c6490db8fe1b533ecda32c21", "results/prereg36/explore/p36_lib.py": "8df27b37a6b3de786b54af25908d3d91ffe4c030dceb7350d9dd130a14ce241f", "results/prereg36/explore/p36_train.py": "8abc5ef22eba35b70be06c624f7071218d0a4266affe5494d1207f8819271211", "results/prereg36/explore/split_record.json": "2c847257139cd73b9a1808c003ba30b86bcd59eb6860edef8043afc82535e7e3", "results/prereg42/explore/p42_embed.py": "23b60a6f71d1aad5881acd10c87ef2c31a9ed81e6ac2d369c3d29615452b69fd", "results/prereg42/explore/p42_lib.py": "b5045f8b8222e1c3e6891675eb4b3296fe4254b974421c71853ea8a43cea6eea", "results/prereg42/explore/p42_train.py": "ec5d7baf7bf16753cd9d0fb7369e3ff4cb5a9a271d8ed28cf264691bb2359c80", "tools/c10data.py": "2906d97fe41171dc955288d0a6e695fcef905417d18d94cb98f146aac74c6495"}
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout)
tv = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available())'], capture_output=True, text=True).stdout.strip()
print('torch', tv)   # in a subprocess: the notebook itself never holds the GPU (p42_train refuses to share it)
assert tv.endswith('True'), 'STOP: 沒有 GPU，請在「變更執行階段類型」選 GPU'
kit = os.path.join(DRIVE, 'C10_p42_kit.zip')
assert os.path.exists(kit), 'STOP: 找不到 ' + kit
ok_flag = os.path.join(WORK, '.kit_ok')
if not (os.path.exists(ok_flag) and open(ok_flag).read() == KIT_SHA):
    assert sha(kit) == KIT_SHA, 'STOP: 套件檔的 SHA-256 不對'
    with zipfile.ZipFile(kit) as z:
        z.extractall(WORK)
    for rel, h in FILES.items():
        assert sha(os.path.join(WORK, rel)) == h, 'STOP: 檔案不對 ' + rel
    open(ok_flag, 'w').write(KIT_SHA)
print('kit ok', len(FILES), 'files')
os.makedirs(os.path.join(DRIVE, VARIANT), exist_ok=True)

In [ ]:
# 2. 訓練（不用改；斷線後重按「全部執行」會從最後的存檔接著跑）
out = os.path.join(DRIVE, VARIANT)
rec = os.path.join(out, 'train_record_%s.json' % VARIANT)
workers = str(max(2, min(12, (os.cpu_count() or 4) - 1)))
env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1', PYTHONIOENCODING='utf-8')
def run(args, log):
    with open(log, 'a') as lf:
        p = subprocess.Popen([sys.executable, '-u'] + args, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='')
            lf.write(line); lf.flush()
        return p.wait()
done = os.path.exists(rec) and json.load(open(rec)).get('finished')
if not done:
    if not any(f.startswith('step_') for f in os.listdir(os.path.join(out, 'ckpt'))) if os.path.isdir(os.path.join(out, 'ckpt')) else True:
        rc = run(['results/prereg42/explore/p42_train.py', '--variant', VARIANT, '--workers', workers, '--bench', '300', '--out', out], os.path.join(out, 'bench_log.txt'))
        print('bench exit', rc)
    rc = run(['results/prereg42/explore/p42_train.py', '--variant', VARIANT, '--workers', workers, '--auto-resume', '--out', out, '--record', rec, '--min-free-ram-mb', '2048'], os.path.join(out, 'train_log.txt'))
    assert rc == 0, 'STOP: 訓練結束碼 %d（4＝記憶體不足或 STOP 檔，重按「全部執行」會接續）' % rc
print('training finished')

In [ ]:
# 3. D 玩家的向量（不用改）
ck = os.path.join(out, 'ckpt', 'step_40000.pt')
emb_root = os.path.join(DRIVE, 'emb')
jobs = [["--weights", "raw"], ["--weights", "ema"]]
for args in jobs:
    rc = run(['results/prereg42/explore/p42_embed.py', '--ckpt', ck, '--players', 'D', '--workers', workers, '--emb-root', emb_root] + args, os.path.join(out, 'embed_log.txt'))
    assert rc == 0, 'STOP: 向量結束碼 %d' % rc
print('embeddings finished')

In [ ]:
# 4. 打包回傳（不用改）
res = os.path.join(DRIVE, 'C10_p42_%s_results.zip' % VARIANT)
with zipfile.ZipFile(res + '.tmp', 'w', zipfile.ZIP_STORED) as z:
    for name in ('train_record_%s.json' % VARIANT, 'train_log.txt', 'train_log.jsonl', 'bench_log.txt', 'embed_log.txt'):
        p = os.path.join(out, name)
        if os.path.exists(p):
            z.write(p, '%s/%s' % (VARIANT, name))
    z.write(ck, '%s/ckpt/step_40000.pt' % VARIANT)
    for d in sorted(os.listdir(emb_root)):
        if d.startswith(VARIANT + '_'):
            z.write(os.path.join(emb_root, d, 'store_D.npz'), 'emb/%s/store_D.npz' % d)
os.replace(res + '.tmp', res)
drive.flush_and_unmount()
print('完成。請下載 Drive 的 C10/p42/' + os.path.basename(res) + ' 回傳給 Claude（%.0f MB）' % (os.path.getsize(res) / 1e6) if os.path.exists(res) else '完成')